# roadrecon — Beat COCO without COCO (B2 → M3 → M2)

Pure, self-contained pothole detection-pretraining on Colab A100. **No COCO weights, no external pretrained model.**
Flow: extract pool → build labeled folds → **30K trial B2** → **kill-gate** → full B2 → M3 anchored → LOSO eval.
See `docs/ROADRECON_QUICKSTART.md`, `docs/METHOD_OPTIONS.md`.


## 1. Setup — clone + install


In [ ]:
!git clone -b feat/natures-labels https://github.com/TahaErr/yolo-contrastive.git /content/yolo-contrastive
%cd /content/yolo-contrastive
!pip -q install -e ".[yolo,pretrain]"   # ultralytics + opencv/pandas/pyarrow (no transformers/geo)


In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## 2. CONFIG


In [ ]:
POOL_TARS_DIR = '/content/drive/MyDrive/SSL_POOL_PARTS'   # TODO: your Drive path (bdd100k/a2d2/mapillary/cityscapes .tar)
POOL_DIR      = '/content/pool_images'
# labeled Pothole-5000 folds are BUILT on Colab in step 3b (absolute /content paths):
LABELED_YAML  = '/content/datasets/splits/cv/logo/fold_0/data.yaml'
FOLD_DIR      = '/content/datasets/splits/cv/logo'
TRIAL_LIMIT = 30000   # 0 = full pool
TRIAL_IMGSZ, TRIAL_EPOCHS = 384, 10
FULL_IMGSZ,  FULL_EPOCHS  = 512, 30
BATCH = 64


## 3. Extract the unlabeled pool


In [ ]:
import os
os.makedirs(POOL_DIR, exist_ok=True)
for t in ['bdd100k','a2d2','mapillary','cityscapes']:
    tar = f'{POOL_TARS_DIR}/{t}.tar'
    if os.path.exists(tar):
        !tar -xf "{tar}" -C "{POOL_DIR}"
print('pool files:', sum(len(fs) for _,_,fs in os.walk(POOL_DIR)))


## 3b. Build the labeled Pothole-5000 folds (for kill-gate + LOSO)
Paste your 10 Roboflow `ds/...?key=...` download URLs below (one per line). They stay in the
runtime only — NOT committed to the repo. Rebuilds the 10 source-disjoint LOGO folds with
**Colab-absolute paths** (so the kill-gate + LOSO resolve; a fold built elsewhere holds foreign
absolute paths that won't work here).


In [ ]:
SOURCES = '''
# paste one Roboflow ds download URL per line, e.g.:
# https://app.roboflow.com/ds/XXXXXXXX?key=YYYYYYYY
'''.strip()

import json
from pathlib import Path
from yolo_contrastive.data.downstream import prepare_downstream, build_cv_splits
assert 'roboflow.com' in SOURCES, 'paste your Roboflow ds URLs into SOURCES first'
Path('/content/sources.txt').write_text(SOURCES + '
')
manifest = prepare_downstream('/content/sources.txt', root='/content/datasets', total=5000, seed=42)
build_cv_splits('/content/datasets/selection_manifest.json', '/content/datasets/splits', scheme='logo', seed=42)
folds = json.loads(Path('/content/datasets/splits/cv/logo/summary.json').read_text())['folds']
print('LOGO folds:', len(folds), '| fold_0 data.yaml:', folds[0]['data_yaml'])


## 4. Stage 1 — 30K TRIAL: B2 pretrain + mine (shared seeded subset)


In [ ]:
!python examples/12_roadrecon_pretrain.py --pool "{POOL_DIR}" --out runs/roadrecon_trial --limit {TRIAL_LIMIT} --imgsz {TRIAL_IMGSZ} --epochs {TRIAL_EPOCHS} --batch {BATCH} --device 0


## 5. KILL-GATE — decide BEFORE the full run
**GO if precision ≥ 0.5 and small_recall ≥ 0.3.** Sweep `--min-box-area` (64/128/256).


In [ ]:
for mba in [64, 128, 256]:
    print('=== min_box_area', mba, '===')
    !python examples/12b_roadrecon_killgate.py --reconstructor runs/roadrecon_trial/roadrecon_full.pt --dataset "{LABELED_YAML}" --min-box-area {mba} --z-thresh 3.0 --iou 0.3 --out runs/killgate_mba{mba}


## 6. Full-pool B2 pretrain + mine (only if the kill-gate said GO)


In [ ]:
!python examples/12_roadrecon_pretrain.py --pool "{POOL_DIR}" --out runs/roadrecon --imgsz {FULL_IMGSZ} --epochs {FULL_EPOCHS} --batch {BATCH} --tap-level P3 --device 0


## 7. M3 — anchored detection-pretraining (scratch, nc=1)
Scratch backbone needs `warmup_steps=0` + higher `backbone_lr` (defaults are COCO-tuned).


In [ ]:
from yolo_contrastive import AnchoredJointTrainer, RoadReconChannel
from yolo_contrastive.roadrecon import build_scratch_detector
trainer = AnchoredJointTrainer(
    model=build_scratch_detector(nc=1),
    replay_data='runs/roadrecon/mined/data.yaml',
    channels=[RoadReconChannel(POOL_DIR, imgsz=FULL_IMGSZ)],
    lambda_aux=1.0, epochs=12, imgsz=FULL_IMGSZ, batch=24,
    warmup_steps=0, backbone_lr=1e-2,
    output_dir='runs/anchored_roadrecon')
M3_CKPT = trainer.train()
print('M3 whole-detector:', M3_CKPT)


## 8. LOSO eval vs COCO & scratch baselines (headline: 10% labels)


In [ ]:
from yolo_contrastive import run_cv_eval
from yolo_contrastive.roadrecon import full_transplant_detection_runner
run_cv_eval(
    [{'name':'roadrecon_m3','backbone_ckpt':M3_CKPT,'base_model':'yolov8n.yaml'}],
    FOLD_DIR, 'runs/cv_roadrecon.csv',
    baselines=('coco','scratch'), fractions=(0.1,0.5,1.0),
    runners={'detection': full_transplant_detection_runner})
